# Experimento 005 - Incompatibilidade de Backbone

Referencias: [Image Classification PyT](https://docs.nvidia.com/tao/tao-toolkit/latest/text/cv_finetuning/pytorch/image_classification_pyt.html), [formato PyTorch](https://docs.nvidia.com/tao/tao-toolkit/latest/text/data_annotation_format.html#image-classification-format-pytorch) e [TAO Toolkit](https://docs.nvidia.com/tao/tao-toolkit/latest/).

Este notebook documenta dois contextos: a configuracao historica com `fan_hybrid_tiny`, que apresentou incompatibilidade, e a configuracao corrigida com `fan_small_12_p4_hybrid`. Nenhum job GPU e criado por padrao.

## Perguntas que o experimento responde

- O backbone historico e compativel com o schema/base experiment?
- O `in_channels` do head corresponde ao backbone escolhido?
- O dataset, workspace e PTM estao corretos?
- A configuracao historica falha no TAO de forma reproduzivel?
- A configuracao corrigida passa pelo treino real?
- Quais metricas, logs e artefatos diferenciam os dois contextos?

## 1. Configuracao comum

`RUN_FAILURE_REPRODUCTION` e `RUN_CORRECTED_TRAINING` permanecem `False` ate a analise de preflight ser revisada.

In [ ]:
import copy
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import requests

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
RUN_FAILURE_REPRODUCTION = False
RUN_CORRECTED_TRAINING = False
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_005_EVIDENCE", "experiment-005-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((item for item in candidates if item.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text())["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY: raise RuntimeError("NGC key not found")
http = requests.Session()
print({"base_url": TAO_BASE_URL, "workspace": WORKSPACE_ID, "dataset": DATASET_URI, "failure_control": RUN_FAILURE_REPRODUCTION, "corrected_control": RUN_CORRECTED_TRAINING})

## 2. Login e preflight comum

Esta etapa nao cria jobs. Ela valida FTMS, workspace, dataset, schema e base experiment antes de comparar os modelos.

In [ ]:
def api(method, path, payload=None, headers=None):
    merged = {"Content-Type": "application/json"}
    if headers: merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=(10, 120))
    response.raise_for_status()
    return response.json()

login = api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
health = api("GET", "/api/v2/health")
workspace = api("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schema = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch=classification_pyt&action=train", headers=AUTH)
base_experiments = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch=classification_pyt", headers=AUTH)
dataset_matches = [x for x in datasets.get("datasets", []) if x.get("cloud_file_path") == "data/tao-test-classification-v1"]
assert "readiness" in health
assert workspace.get("id") == WORKSPACE_ID
assert dataset_matches
assert any(x.get("id") == BASE_EXPERIMENT_ID for x in base_experiments.get("experiments", []))
print("common preflight=PASS dataset=", dataset_matches[0].get("name"), "base=", BASE_EXPERIMENT_ID)

## 3. Comparacao de compatibilidade

O contexto historico usa `fan_hybrid_tiny`. O contexto corrigido usa `fan_small_12_p4_hybrid` com head `in_channels=448`, alinhado ao modelo que passou nos experimentos posteriores.

In [ ]:
default_spec = copy.deepcopy(schema.get("default", {}))
historical = copy.deepcopy(default_spec)
historical.setdefault("model", {}).setdefault("backbone", {})["type"] = "fan_hybrid_tiny"
corrected = copy.deepcopy(default_spec)
corrected.setdefault("model", {}).setdefault("backbone", {})["type"] = "fan_small_12_p4_hybrid"
corrected.setdefault("model", {}).setdefault("head", {})["in_channels"] = 448
for spec in (historical, corrected):
    spec.setdefault("train", {}).update({"num_gpus": 1, "num_epochs": 1})
    spec.setdefault("dataset", {}).update({"root_dir": DATASET_URI, "num_classes": 2})
comparison = {"historical": historical, "corrected": corrected, "base_metadata": [x for x in base_experiments.get("experiments", []) if x.get("id") == BASE_EXPERIMENT_ID]}
(EVIDENCE_DIR / "compatibility-comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps({"historical_backbone": historical.get("model", {}).get("backbone", {}).get("type"), "corrected_backbone": corrected.get("model", {}).get("backbone", {}).get("type"), "corrected_in_channels": corrected.get("model", {}).get("head", {}).get("in_channels")}, indent=2))

## 4. Contexto historico - reproduzir falha

Este bloco somente cria um job se `RUN_FAILURE_REPRODUCTION=True`. O resultado esperado e uma falha terminal documentada por incompatibilidade, nao um erro silencioso.

In [ ]:
def make_payload(name, specs):
    return {"kind": "experiment", "name": name, "network_arch": "classification_pyt", "encryption_key": "unittest-only", "workspace": WORKSPACE_ID, "action": "train", "specs": specs, "train_dataset_uris": [DATASET_URI], "eval_dataset_uri": DATASET_URI, "inference_dataset_uri": DATASET_URI, "base_experiment_ids": [BASE_EXPERIMENT_ID]}

def poll(job_id, label):
    for _ in range(360):
        current = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}", headers=AUTH)
        (EVIDENCE_DIR / f"{label}-status.json").write_text(json.dumps(current, indent=2))
        state = current.get("status")
        print(label, state)
        if state in {"Done", "Completed", "Error", "Failed", "Failure", "Canceled", "Cancelled"}: return current
        time.sleep(10)
    raise TimeoutError(job_id)

failure_result = {"executed": False, "expected": "terminal incompatibility failure"}
if RUN_FAILURE_REPRODUCTION:
    payload = make_payload(f"experiment-005-historical-{int(time.time())}", historical)
    (EVIDENCE_DIR / "historical-payload.json").write_text(json.dumps(payload, indent=2))
    submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    failure_job_id = submitted["id"]
    failure_status = poll(failure_job_id, "historical")
    failure_result = {"executed": True, "job_id": failure_job_id, "status": failure_status.get("status"), "job_details": failure_status.get("job_details")}
    (EVIDENCE_DIR / "historical-result.json").write_text(json.dumps(failure_result, indent=2))
print(json.dumps(failure_result, indent=2))

## 5. Contexto corrigido - validar sucesso

Este bloco somente cria um job se `RUN_CORRECTED_TRAINING=True`. O resultado esperado e `Done`, `SUCCESS`, checkpoint e metricas.

In [ ]:
corrected_result = {"executed": False, "expected": "Done with SUCCESS and metrics"}
if RUN_CORRECTED_TRAINING:
    payload = make_payload(f"experiment-005-corrected-{int(time.time())}", corrected)
    (EVIDENCE_DIR / "corrected-payload.json").write_text(json.dumps(payload, indent=2))
    submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    corrected_job_id = submitted["id"]
    corrected_status = poll(corrected_job_id, "corrected")
    corrected_result = {"executed": True, "job_id": corrected_job_id, "status": corrected_status.get("status"), "job_details": corrected_status.get("job_details")}
    (EVIDENCE_DIR / "corrected-result.json").write_text(json.dumps(corrected_result, indent=2))
print(json.dumps(corrected_result, indent=2))

## 6. Checklist final

O notebook separa evidencias de preflight, reproducao da falha e validacao corrigida. `NOT_RUN` significa que o controle GPU estava desligado; nao significa sucesso.

In [ ]:
checks = {
    "login_health_workspace_dataset_schema": True,
    "dataset_and_base_preflight": bool(dataset_matches) and preflight.get("base_experiment_found", False),
    "historical_context_documented": True,
    "corrected_context_documented": True,
    "historical_failure_reproduced": failure_result.get("status") in {"Error", "Failed", "Failure"} if failure_result.get("executed") else None,
    "corrected_training_passed": corrected_result.get("status") in {"Done", "Completed"} if corrected_result.get("executed") else None,
}
report = {"experiment": "005-training-real", "checks": checks, "evidence_files": sorted(p.name for p in EVIDENCE_DIR.iterdir()), "generated_at": datetime.now(timezone.utc).isoformat()}
(EVIDENCE_DIR / "final-checklist.json").write_text(json.dumps(report, indent=2))
for name, value in checks.items(): print(f"{name}: {value if value is not None else 'NOT_RUN'}")
print("Evidence directory:", EVIDENCE_DIR)